In [1]:
MODEL_PATH = "/home/jjs2403/2026_bootcamp_summer/models/Qwen3-VL-8B-Instruct"
IMAGE_DIR = "/home/jjs2403/2026_bootcamp_summer/_ASSIGNMENTS/competitions/competition1/images"
SAMPLE_SUB_CSV = "sample_submission.csv"
OUTPUT_CSV = "submission.csv"
RESIZE_SIZE = 840
SEED = 42
N_CROPS = 3
FINAL_MAX_NEW_TOKENS = 1 # A,B,C... 분류를 위한 토큰 제한

In [2]:
import os
import re
import random
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

In [3]:
# Seed 고정용
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
def load_qwen_model(model_path: str):
    # Processor
    processor = AutoProcessor.from_pretrained(model_path)
    print(f"Loading Qwen model from {model_path}...")

    # model
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_path,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )

    print("Model loaded successfully!")
    return model, processor

model, processor = load_qwen_model(MODEL_PATH)

Loading Qwen model from /home/jjs2403/2026_bootcamp_summer/models/Qwen3-VL-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
def normalize_bbox(bbox: List[float], img_width: int, img_height: int) -> List[int]:
    x1, y1, x2, y2 = bbox
    if max(bbox) <= 1000:
        x1 = int(x1 * img_width / 1000)
        y1 = int(y1 * img_height / 1000)
        x2 = int(x2 * img_width / 1000)
        y2 = int(y2 * img_height / 1000)
    else:
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    return [x1, y1, x2, y2]


def crop_image_with_bbox(image: Image.Image, bbox: List[int], padding: float = 0.1) -> Image.Image:
    """바운딩 박스 영역을 이미지에서 추출"""
    x1, y1, x2, y2 = bbox

    width = x2 - x1
    height = y2 - y1

    pad_w = int(width * padding)
    pad_h = int(height * padding)

    x1 = max(0, x1 - pad_w)
    y1 = max(0, y1 - pad_h)
    x2 = min(image.width, x2 + pad_w)
    y2 = min(image.height, y2 + pad_h)

    # 최소 크기 보장 (224x224)
    crop_width = x2 - x1
    crop_height = y2 - y1

    if crop_width < 224 or crop_height < 224:
        # bbox 중심점 계산
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2

        # 최소 크기 224 보장
        half_size = max(crop_width, crop_height, 224) // 2

        # 중심점 기준으로 정사각형 영역 설정
        x1 = max(0, center_x - half_size)
        y1 = max(0, center_y - half_size)
        x2 = min(image.width, center_x + half_size)
        y2 = min(image.height, center_y + half_size)

    # PIL의 crop 함수로 이미지 잘라내기
    return image.crop((x1, y1, x2, y2))


# 프롬프트로 유도한 [x1, y1, x2, y2] 추출하기 위함
# Qwen-VL이 프롬프트와 무관하게 <|box_start|>(123,456),(789,123)<|box_end|> 형식으로 때때로 출력(이를 추출하기 위함)
BOX_PATTERN = r'<\|box_start\|>\s*\(([\d.]+),\s*([\d.]+)\),\s*\(([\d.]+),\s*([\d.]+)\)\s*<\|box_end\|>'
JSON_PATTERN = r'\[\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)\s*\]'


def extract_bboxes_from_text(text: str, max_n: int = N_CROPS) -> List[List[float]]:

    boxes = [[float(v) for v in m] for m in re.findall(BOX_PATTERN, text)]

    if not boxes:
        boxes = [[float(v) for v in m] for m in re.findall(JSON_PATTERN, text)]

    return boxes[:max_n]

In [ ]:

ANSWER_PATTERN = r'Answer\s*:\s*([A-Ha-h])\b'

# 프롬프트로 유도된 출력 추출
def extract_label(text: str) -> Optional[str]:

    tail = text.split("assistant")[-1].strip()

    matches = re.findall(ANSWER_PATTERN, tail, re.IGNORECASE)
    if matches:
        return matches[-1].upper()
    
    if len(tail) == 1 and tail.upper() in "ABCDEFGH":
        return tail.upper()

    return None

In [ ]:

def run_qwen_inference(model, processor, images, prompt, max_new_tokens=256):
    content = []

    for img in images:
        resized = img.resize((RESIZE_SIZE, RESIZE_SIZE), Image.BILINEAR)
        content.append({"type": "image", "image": resized})

    content.append({"type": "text", "text": prompt})

    message = [{"role": "user", "content": content}]

    chat_prompt = processor.apply_chat_template(
        message,
        tokenize=False,             
        add_generation_prompt=True  
    )


    resized_images = [img.resize((RESIZE_SIZE, RESIZE_SIZE), Image.BILINEAR) for img in images]
    inputs = processor(
        text=[chat_prompt],   
        images=resized_images,  
        return_tensors="pt"     
    )

    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}


    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,             
            use_cache=True,    
        )


    trimmed = generated[:, inputs["input_ids"].shape[-1]:]

    response = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return response.strip()


In [ ]:

TURN1_PROMPT = """You are a visual reasoning assistant helping to locate informative regions in an image.

이 이미지에 나타난 매장의 카테고리로 가장 적절한 것을 고르시오.
A. 한식
B. 양식
C. 일식
D. 중식
E. 카페
F. 베이커리
G. 주점
H. 동남아

Identify exactly 3 regions in the image that contain the most useful visual evidence for determining the store's category (for example: readable text or signage, menu displays, storefront design, or distinctive interior details).

Requirements:
- You must output exactly 3 bounding boxes. Never output fewer than 3.
- If you cannot find 3 clearly distinct informative regions, still provide 3 boxes by including the next most useful regions available (e.g. a secondary storefront detail). Do not skip, merge, or leave any box blank.
- Each bounding box must tightly enclose only the relevant region. Do not include large areas of irrelevant background (sky, street, pedestrians, unrelated neighboring stores).
- Coordinates must be integers in the range [0, 1000], in the format [x1, y1, x2, y2].
- Output only the 3 lines below. Do not include any reasoning, explanation, or extra text.

Bounding Box 1: [x1, y1, x2, y2]
Bounding Box 2: [x1, y1, x2, y2]
Bounding Box 3: [x1, y1, x2, y2]"""



TURN2_PROMPT = """You are a visual reasoning assistant. You are given several images:

1. The first image is the original full image (for context)
2. The remaining images, if any, are cropped detail regions identified for answering the question

이 이미지에 나타난 매장의 카테고리로 가장 적절한 것을 고르시오.
A. 한식
B. 양식
C. 일식
D. 중식
E. 카페
F. 베이커리
G. 주점
H. 동남아

Consider the original image for overall context and the cropped regions for detailed visual evidence. If a cropped region does not contain useful evidence (e.g. only background or an unrelated object), disregard it and rely on the other images instead.

Analyze the visual evidence in all provided images, including:
- visible text and signage (store name, menu items, or other category-indicating words)
- storefront and interior appearance
- lighting, decorations, and overall atmosphere
- menus or other displayed information
- objects, food, tableware, and other visually distinctive elements

Prioritize evidence roughly in this order: (1) legible text such as the store name or menu items, (2) visible food, products, or objects, (3) overall interior and atmosphere. Use text evidence when it is clearly readable; otherwise weigh the remaining visual cues together.

Briefly reason step by step: note the strongest 1-2 pieces of evidence you observe, identify which categories that evidence is most consistent with, and choose the single best match. If the evidence seems ambiguous between two categories, choose the one with the stronger overall match rather than relying on a single superficial cue.

Even if the visual evidence is ambiguous or insufficient, you must select the single most appropriate option from A, B, C, D, E, F, G, or H.

Then end your response with a single line in exactly this format:

Answer: X

where X is one of A, B, C, D, E, F, G, or H. Write nothing after that line."""


FINAL_PROMPT = """이 이미지에 나타난 매장의 카테고리로 가장 적절한 것을 고르시오.
A. 한식
B. 양식
C. 일식
D. 중식
E. 카페
F. 베이커리
G. 주점
H. 동남아

Please answer the question based on the image.

Rules for your response:
- Your entire response must be exactly one character.
- That character must be one of: A B C D E F G H
- Do not write any explanation, reasoning, punctuation, or extra whitespace.
- If you are uncertain, you must still choose the single most likely option.

Answer:"""



===== TURN1_PROMPT =====
You are a visual reasoning assistant helping to locate informative regions in an image.

이 이미지에 나타난 매장의 카테고리로 가장 적절한 것을 고르시오.
A. 한식
B. 양식
C. 일식
D. 중식
E. 카페
F. 베이커리
G. 주점
H. 동남아

Identify exactly 3 regions in the image that contain the most useful visual evidence for determining the store's category (for example: readable text or signage, menu displays, storefront design, or distinctive interior details).

Requirements:
- You must output exactly 3 bounding boxes. Never output fewer than 3.
- If you cannot find 3 clearly distinct informative regions, still provide 3 boxes by including the next most useful regions available (e.g. a secondary storefront detail). Do not skip, merge, or leave any box blank.
- Each bounding box must tightly enclose only the relevant region. Do not include large areas of irrelevant background (sky, street, pedestrians, unrelated neighboring stores).
- Coordinates must be integers in the range [0, 1000], in the format [x1, y1, x2, y2].


In [ ]:

set_seed(SEED)


submission = pd.read_csv(SAMPLE_SUB_CSV)
label_col = submission.columns[1] 
print(f"submission columns: {list(submission.columns)} / 라벨 컬럼: '{label_col}'")


image_ids = submission["ID"].tolist()
image_paths = [os.path.join(IMAGE_DIR, i) for i in image_ids]

existing_files = {entry.name for entry in os.scandir(IMAGE_DIR)}



preds = []



for image_id, image_path in tqdm(list(zip(image_ids, image_paths)), total=len(image_ids)):
    
    with Image.open(image_path) as image:
        image = image.convert("RGB")

    # bbox 좌표를 최대 3개 예측
    turn1_output = run_qwen_inference(
        model, processor,
        [image],
        TURN1_PROMPT,
        max_new_tokens=256
    )
    bboxes = extract_bboxes_from_text(turn1_output, N_CROPS)

    if len(bboxes) < N_CROPS:
        n_bbox_short += 1

    # bbox 좌표로 crop 이미지 생성
    cropped_images = []
    for bbox in bboxes:
        pixel_bbox = normalize_bbox(bbox, image.width, image.height)
        cropped_images.append(crop_image_with_bbox(image, pixel_bbox, padding=0.1))

    # 원본과 crop 이미지를 프롬프트와 함께 넣고 분류
    turn2_output = run_qwen_inference(
        model, processor,
        [image] + cropped_images,
        TURN2_PROMPT,
        max_new_tokens=512
    )
    label = extract_label(turn2_output)

    # 분류 실패 시 프롬프트로 분류 강제
    if label is None:
        final_output = run_qwen_inference(
            model, processor,
            [image],
            FINAL_PROMPT,
            max_new_tokens=FINAL_MAX_NEW_TOKENS
        )
        label = extract_label(final_output)


    preds.append(label)



submission columns: ['ID', 'label'] / 라벨 컬럼: 'label'
이미지 200장 경로 확인 완료


100%|██████████| 200/200 [32:17<00:00,  9.69s/it]


bbox 3개 미만이었던 이미지: 0 / 200
crop 단계 파싱 실패 → 원본 단계로 내려감: 1
원본 단계에서도 실패: 0 (0 이어야 정상)


In [ ]:

submission[label_col] = preds
submission.to_csv(OUTPUT_CSV, index=False)

print(f"saved: {OUTPUT_CSV}")
print(submission.head())


saved: submission.csv
           ID label
0  000001.jpg     E
1  000002.jpg     A
2  000003.jpg     E
3  000004.jpg     E
4  000005.jpg     B

행 수: 200 (기대: 200)
결측: 0 (기대: 0)
ID 중복: 0 (기대: 0)
라벨 분포:
label
A    48
B    26
C    23
D    21
E    32
F    16
G    20
H    14
Name: count, dtype: int64

형식 위반 행: 0 (기대: 0)
